# AcentoPartners Email Classifier — Notebook explicativo

Este notebook recorre **paso a paso** toda la lógica del clasificador de emails, sin FastAPI ni frontend.  
Puedes ejecutarlo celda por celda para entender exactamente qué hace cada componente.

---

## ¿Qué problema resuelve?

AcentoPartners recibe emails de prestamistas (JLL, Capital One, M&T Bank, etc.) exigiendo documentos  
de cumplimiento de seguros. Actualmente un operador lee cada email manualmente y decide:
- ¿Qué prestamista envió esto?
- ¿Qué tipo de deficiencia reclaman?
- ¿Qué documentos hay que juntar y enviar?

**El sistema automatiza esa clasificación** usando un LLM local (Ollama).

## Flujo completo

```
📧 Email (.eml)  →  📋 Parser  →  🔍 Detección de dominio  →  🤖 LLM  →  📚 Knowledge Base  →  💾 DB
```

## Requisitos para correr este notebook

| Componente | Requerido | Alternativa si no está |
|---|---|---|
| `mail-parser` | Sí | — |
| `ollama` (Python SDK) | Sí | El notebook tiene un mock automático |
| Ollama corriendo localmente | No | Se usa respuesta simulada |
| Archivos `.eml` de muestra | Sí | Incluidos en `sample_emails/` |

---
## Paso 0 — Instalar dependencias

Solo necesitamos `mail-parser` para leer archivos `.eml` y `ollama` para el LLM.  
Si Ollama no está corriendo, el notebook usa una respuesta simulada automáticamente.

In [ ]:
# Instalar dependencias (solo necesario la primera vez)
import subprocess, sys

packages = ["mail-parser", "ollama"]
for pkg in packages:
    result = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], capture_output=True)
    status = "✅" if result.returncode == 0 else "❌"
    print(f"{status} {pkg}")

---
## Paso 1 — La Knowledge Base (Base de conocimiento)

Este es el corazón del sistema. Es una tabla que define:
- Qué **prestamistas** existen (JLL, Capital One, M&T Bank, etc.)
- Qué **tipos de waiver** puede reclamar cada uno
- Qué **documentos** hay que juntar para responder
- Cómo se reconoce cada prestamista por **dominio de email**

En el código de producción esto está en `app/core/knowledge_base.py`.  
Aquí lo definimos directamente para entenderlo sin imports.

In [ ]:
# ============================================================
# KNOWLEDGE BASE — Matriz prestamista / tipo de waiver
# Cada entrada define UN prestamista + UN tipo de problema
# ============================================================

LENDER_WAIVER_MATRIX = [
    {
        "lender": "JLL (Insurance Servicing)",
        "lender_aliases": ["JLL", "Jones Lang LaSalle", "JLL Insurance"],
        "waiver_type": "Assault & Battery (A&B) sublimit",
        "triggers": "GL tiene sublímite de A&B (ej. $250k) menor al requerido por JLL.",
        "evidence_required_ops": "Cámaras, acceso controlado, guardias de seguridad, antecedentes.",
        "evidence_required_insurance": "Endorsement GL con A&B, lista de rechazos, loss runs.",
        "documents_expected": "ACORD 25 + 101, endorsement A&B, declination letters, loss runs.",
        "waiver_pack": "ACORD 25+101, GL A&B endorsement, Declination letters, Loss runs, Security Fact Sheet",
        "actions_to_automate": "Security Fact Sheet por propiedad, pre-adjuntar declinations + loss runs.",
    },
    {
        "lender": "JLL (Insurance Servicing)",
        "lender_aliases": ["JLL", "Jones Lang LaSalle", "JLL Insurance"],
        "waiver_type": "Sexual Abuse & Molestation (SAM)",
        "triggers": "GL/excess no cubre SAM o está excluido.",
        "evidence_required_ops": "Seguridad, background checks, cámaras en áreas comunes.",
        "evidence_required_insurance": "Póliza SAM standalone ($2M), endorsements GL.",
        "documents_expected": "ACORD 25+101, copia de póliza SAM, loss runs, resumen seguridad.",
        "waiver_pack": "ACORD 25+101, Standalone SAM policy, Loss runs, Security controls",
        "actions_to_automate": "Mostrar SAM en ACORD 101, adjuntar póliza SAM.",
    },
    {
        "lender": "JLL (Insurance Servicing)",
        "lender_aliases": ["JLL", "Jones Lang LaSalle", "JLL Insurance"],
        "waiver_type": "Equipment Breakdown (EB) limit",
        "triggers": "JLL requiere EB = 100% del valor asegurado; $1M es insuficiente.",
        "evidence_required_ops": "Memo de ingeniería: equipo central vs distribuido, capacidades.",
        "evidence_required_insurance": "Endorsement EB, SOV con activos relevantes.",
        "documents_expected": "ACORD 28+101, SOV, memo de ingeniería, endorsement EB.",
        "waiver_pack": "ACORD 28+101, SOV equipment pages, Engineering memo, EB endorsement",
        "actions_to_automate": "Matriz EB por propiedad, auto-generar memo desde datos de construcción.",
    },
    {
        "lender": "Capital One (Servicing)",
        "lender_aliases": ["Capital One", "CapOne", "Capital One Servicing"],
        "waiver_type": "Full Policy Package timing",
        "triggers": "Capital One emite Non-Compliance si documentos no llegan en 90 días de renovación.",
        "evidence_required_ops": "Correcta redacción de mortgagee clause, dirección de propiedad.",
        "evidence_required_insurance": "Póliza completa GL/Umbrella/Property/Terror, cláusula AI corregida, SOV.",
        "documents_expected": "ACORD 25/28, SOV, carta compromiso entrega en 90 días.",
        "waiver_pack": "ACORD 25+28+101, SOV, AI clause (Capital One/Freddie wording), Policy delivery letter",
        "actions_to_automate": "Tracker de entrega de pólizas, auto-enviar COIs + nota de compromiso.",
    },
    {
        "lender": "Freddie Mac (via JLL Real Estate Capital)",
        "lender_aliases": ["Freddie Mac", "Freddie", "JLL Real Estate Capital", "FHLMC"],
        "waiver_type": "Additional Insured/Mortgagee wording",
        "triggers": "Freddie exige wording exacto ISAOA/ATIMA; wording incorrecto genera deficiencia.",
        "evidence_required_ops": "No aplica; es problema de documentación.",
        "evidence_required_insurance": "COIs con cláusula AI de Freddie, terrorismo en ACORD 101.",
        "documents_expected": "COIs corregidos, SOV, nota de terrorismo, log de cambios.",
        "waiver_pack": "COI con Freddie wording, SOV, ACORD 101 Terrorism note",
        "actions_to_automate": "Mantener texto Freddie en biblioteca, pre-validar en cada COI.",
    },
    {
        "lender": "M&T Bank",
        "lender_aliases": ["M&T", "M&T Bank", "MTB", "mtb.com"],
        "waiver_type": "Multi-issue compliance (A&B / SAM / BI / OL)",
        "triggers": "M&T emite First/Second Notice of Non-Compliance con múltiples requerimientos.",
        "evidence_required_ops": "Seguridad por propiedad, periodo de espera BI (max 72hr), deducibles.",
        "evidence_required_insurance": "ACORD 25+28+101, GL A&B, SAM standalone, BI/OL, Terrorismo.",
        "documents_expected": "ACORD 25+28+101, SOV, A&B endorsement, SAM policy, BI/OL endorsements.",
        "waiver_pack": "ACORD 25/28/101, SOV, A&B endorsement, SAM policy, BI/OL endorsements",
        "actions_to_automate": "Template multi-issue, pre-llenar detalles A&B, adjuntar SAM, confirmar BI/OL.",
    },
    {
        "lender": "Berkadia",
        "lender_aliases": ["Berkadia"],
        "waiver_type": "Invoice components (Excess/Terrorism) & Address",
        "triggers": "Berkadia necesita facturas con líneas separadas de Terrorismo y Excess, y dirección correcta.",
        "evidence_required_ops": "Confirmar dirección física correcta, instrucciones de escrow.",
        "evidence_required_insurance": "Factura con líneas Property/GL/Umbrella/Excess/Terrorism, ACORDs.",
        "documents_expected": "Factura + ACORD 25/28, extracto SOV con dirección, recibo de pago.",
        "waiver_pack": "Invoice with component flags, ACORD 25/28, SOV address rows",
        "actions_to_automate": "Template de factura con líneas requeridas, adjuntar SOV automáticamente.",
    },
    {
        "lender": "Greystone",
        "lender_aliases": ["Greystone"],
        "waiver_type": "ACORD-gate for payment & Umbrella clarity",
        "triggers": "Greystone necesita ACORD 25/28 antes de liberar pago y claridad sobre Umbrella.",
        "evidence_required_ops": "Confirmar contactos de pago, alineación factura/COI.",
        "evidence_required_insurance": "ACORD 25/28, factura con GL vs Umbrella separados.",
        "documents_expected": "ACORD 25/28 con cada factura, GL vs Umbrella en factura.",
        "waiver_pack": "ACORD 25/28, Invoice with GL/Umbrella breakout",
        "actions_to_automate": "Adjuntar ACORD 25/28 automáticamente con cada factura.",
    },
]

# Mapa de dominio de email → nombre de prestamista
DOMAIN_LENDER_MAP = {
    "jll.com": "JLL (Insurance Servicing)",
    "am.jll.com": "JLL (Insurance Servicing)",
    "cmservicing.com": "Capital One (Servicing)",
    "capitalone.com": "Capital One (Servicing)",
    "freddiemac.com": "Freddie Mac (via JLL Real Estate Capital)",
    "grandbridge.com": "Grandbridge / KeyBank / Wells Fargo Trustee chain",
    "keybank.com": "Grandbridge / KeyBank / Wells Fargo Trustee chain",
    "berkadia.com": "Berkadia",
    "nmrk.com": "NEWMARK (MCM Servicing)",
    "newmark.com": "NEWMARK (MCM Servicing)",
    "greystone.com": "Greystone",
    "cbre.com": "CBRE (Insurance Servicing)",
    "mtb.com": "M&T Bank",
}

print(f"✅ Knowledge Base cargada: {len(LENDER_WAIVER_MATRIX)} entradas")
print(f"✅ Mapa de dominios: {len(DOMAIN_LENDER_MAP)} dominios registrados")
print()
print("Prestamistas disponibles:")
lenders_unique = list(dict.fromkeys(e['lender'] for e in LENDER_WAIVER_MATRIX))
for l in lenders_unique:
    count = sum(1 for e in LENDER_WAIVER_MATRIX if e['lender'] == l)
    print(f"  • {l} ({count} waiver type{'s' if count > 1 else ''})")

---
## Paso 2 — Funciones auxiliares de la Knowledge Base

Tres funciones clave que usa el sistema:  
1. `find_matching_entry()` — dado un prestamista y waiver, busca los detalles en la matriz  
2. `get_knowledge_base_text()` — genera el texto que se inyecta en el prompt del LLM  
3. `identify_lender_from_domains()` — detecta el prestamista antes de llamar al LLM

In [ ]:
def find_matching_entry(lender: str, waiver_type: str) -> dict | None:
    """
    Busca en la Knowledge Base la entrada que coincide con
    el prestamista y tipo de waiver identificados por el LLM.
    Primero busca match exacto (lender + waiver), luego solo por lender.
    """
    lender_lower = lender.lower()
    waiver_lower = waiver_type.lower()

    # Intento 1: match completo (prestamista + tipo de waiver)
    for entry in LENDER_WAIVER_MATRIX:
        lender_match = (
            lender_lower in entry["lender"].lower()
            or any(lender_lower in alias.lower() for alias in entry["lender_aliases"])
        )
        waiver_match = waiver_lower in entry["waiver_type"].lower()
        if lender_match and waiver_match:
            return entry

    # Intento 2: solo por prestamista (si el waiver no coincide exacto)
    for entry in LENDER_WAIVER_MATRIX:
        lender_match = (
            lender_lower in entry["lender"].lower()
            or any(lender_lower in alias.lower() for alias in entry["lender_aliases"])
        )
        if lender_match:
            return entry

    return None


def get_knowledge_base_text() -> str:
    """
    Genera el texto de la Knowledge Base para inyectar en el prompt del LLM.
    El LLM usa este texto como contexto para clasificar.
    """
    lines = ["=== BASE DE CONOCIMIENTO PRESTAMISTA/WAIVER ===\n"]
    for i, entry in enumerate(LENDER_WAIVER_MATRIX, 1):
        lines.append(f"--- Entrada {i} ---")
        lines.append(f"Prestamista: {entry['lender']}")
        lines.append(f"También conocido como: {', '.join(entry['lender_aliases'])}")
        lines.append(f"Tipo de Waiver: {entry['waiver_type']}")
        lines.append(f"Qué lo dispara: {entry['triggers']}")
        lines.append(f"Documentos esperados: {entry['documents_expected']}")
        lines.append("")
    return "\n".join(lines)


def identify_lender_from_domains(
    from_domain: str,
    to_domains: list[str],
    cc_domains: list[str],
) -> tuple[str | None, str]:
    """
    Detecta el prestamista ANTES de llamar al LLM, solo usando dominios de email.
    
    Prioridad: TO > CC > FROM
    ¿Por qué? Estos emails son RESPUESTAS del agente de seguros AL prestamista,
    entonces el prestamista suele estar en el campo TO, no en el FROM.
    """
    # Dominios internos que no son prestamistas
    internal_domains = {"acentopartners.com", "captiveadvisorypartners.com"}

    # 1. Revisar TO primero (mayor prioridad)
    for domain in to_domains:
        if domain not in internal_domains and domain in DOMAIN_LENDER_MAP:
            return DOMAIN_LENDER_MAP[domain], f"dominio TO: {domain}"

    # 2. Revisar CC
    for domain in cc_domains:
        if domain not in internal_domains and domain in DOMAIN_LENDER_MAP:
            return DOMAIN_LENDER_MAP[domain], f"dominio CC: {domain}"

    # 3. Revisar FROM (menor prioridad)
    if from_domain and from_domain not in internal_domains:
        if from_domain in DOMAIN_LENDER_MAP:
            return DOMAIN_LENDER_MAP[from_domain], f"dominio FROM: {from_domain}"

    return None, "sin coincidencia de dominio"


# --- Prueba rápida ---
print("🧪 Prueba de detección de dominio:")
lender, source = identify_lender_from_domains(
    from_domain="captiveadvisorypartners.com",   # agente de seguros (interno)
    to_domains=["jll.com"],                       # JLL es el destinatario
    cc_domains=["acentopartners.com"],
)
print(f"  Detectado: '{lender}' via {source}")
print()

print("🧪 Prueba de búsqueda en Knowledge Base:")
entry = find_matching_entry("JLL", "A&B")
if entry:
    print(f"  Prestamista: {entry['lender']}")
    print(f"  Waiver: {entry['waiver_type']}")
    print(f"  WaiverPack: {entry['waiver_pack']}")

---
## Paso 3 — Parsear un email `.eml`

Los archivos `.eml` son emails guardados en formato texto estándar.  
El parser extrae los campos importantes: remitente, destinatarios, asunto, cuerpo y adjuntos.

> **¿Por qué importa TO y CC?**  
> El clasificador da prioridad al dominio en el campo TO para identificar al prestamista,  
> porque estos emails son **respuestas** del agente de seguros hacia el prestamista.

In [ ]:
import mailparser
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, field

@dataclass
class EmailData:
    """Estructura de datos de un email ya parseado."""
    source: str                          # 'file' o 'outlook'
    filename: str = ""
    message_id: str = ""
    subject: str = "(sin asunto)"
    sender: str = ""
    sender_domain: str = ""
    to_recipients: list = field(default_factory=list)
    to_domains: list = field(default_factory=list)
    cc_recipients: list = field(default_factory=list)
    cc_domains: list = field(default_factory=list)
    received_date: datetime = None
    body_text: str = ""
    has_attachments: bool = False
    attachment_names: list = field(default_factory=list)


def parse_eml_file(file_path: Path) -> EmailData:
    """Parsea un archivo .eml y retorna un EmailData."""
    mail = mailparser.parse_from_file(str(file_path))

    # Extraer remitente
    sender = mail.from_[0][1] if mail.from_ else ""
    sender_domain = sender.split("@")[1].lower() if "@" in sender else ""

    # Extraer TO: lista de (nombre, email)
    to_emails = [r[1] for r in (mail.to_ or []) if r[1]]
    to_domains = list(dict.fromkeys(
        e.split("@")[1].lower() for e in to_emails if "@" in e
    ))

    # Extraer CC
    cc_emails = [r[1] for r in (mail.cc_ or []) if r[1]]
    cc_domains = list(dict.fromkeys(
        e.split("@")[1].lower() for e in cc_emails if "@" in e
    ))

    # Cuerpo: preferir texto plano, si no existe usar HTML
    body_text = mail.text_plain[0] if mail.text_plain else ""
    if not body_text and mail.text_html:
        body_text = mail.text_html_no_urls[0] if mail.text_html_no_urls else mail.text_html[0]

    # Adjuntos (excluir imágenes inline)
    attachment_names = [
        att.get("filename", "")
        for att in mail.attachments
        if att.get("filename") and not att["filename"].startswith("image")
    ]

    # Fecha recibida
    received_date = None
    if mail.date:
        received_date = mail.date if isinstance(mail.date, datetime) else datetime.now(timezone.utc)

    return EmailData(
        source="file",
        filename=file_path.name,
        message_id=mail.message_id or "",
        subject=mail.subject or "(sin asunto)",
        sender=sender,
        sender_domain=sender_domain,
        to_recipients=to_emails,
        to_domains=to_domains,
        cc_recipients=cc_emails,
        cc_domains=cc_domains,
        received_date=received_date,
        body_text=body_text.strip(),
        has_attachments=len(attachment_names) > 0,
        attachment_names=attachment_names,
    )


# --- Buscar archivos .eml disponibles ---
sample_dir = Path("sample_emails")
eml_files = sorted(sample_dir.glob("*.eml"))

print(f"📁 Archivos .eml encontrados: {len(eml_files)}")
for f in eml_files:
    print(f"  • {f.name}")

In [ ]:
# Parsear el primer email y mostrar sus campos
if eml_files:
    email = parse_eml_file(eml_files[0])

    print("=" * 60)
    print(f"📧 Archivo:      {email.filename}")
    print(f"📌 Asunto:       {email.subject}")
    print(f"📤 De:           {email.sender}")
    print(f"   Dominio FROM: {email.sender_domain}")
    print(f"📥 Para:         {', '.join(email.to_recipients[:3]) or '(vacío)'}")
    print(f"   Dominios TO:  {email.to_domains}")
    print(f"📋 CC:           {', '.join(email.cc_recipients[:3]) or '(vacío)'}")
    print(f"   Dominios CC:  {email.cc_domains}")
    print(f"📎 Adjuntos:     {email.attachment_names or 'ninguno'}")
    print(f"📅 Fecha:        {email.received_date}")
    print("=" * 60)
    print("📄 Cuerpo (primeros 500 caracteres):")
    print(email.body_text[:500])
    print("...")
else:
    print("❌ No se encontraron archivos .eml en sample_emails/")

---
## Paso 4 — Detección del prestamista por dominio (antes del LLM)

Antes de enviar el email al LLM, el sistema intenta identificar el prestamista  
simplemente mirando los dominios de email (TO, CC, FROM).

Esto sirve como **pista fuerte** que se inyecta en el prompt del LLM.  
Si el dominio coincide exactamente, el LLM casi siempre confirma esa pista.

In [ ]:
# Aplicar detección de dominio al email parseado
if eml_files:
    domain_hint, hint_source = identify_lender_from_domains(
        from_domain=email.sender_domain,
        to_domains=email.to_domains,
        cc_domains=email.cc_domains,
    )

    print("🔍 Resultado de detección por dominio:")
    print(f"   Prestamista detectado: {domain_hint or 'DESCONOCIDO'}")
    print(f"   Fuente:                {hint_source}")
    print()

    if domain_hint:
        print("✅ Pista fuerte disponible — se inyectará en el prompt del LLM")
    else:
        print("⚠️  Sin pista de dominio — el LLM deberá identificar el prestamista solo por el contenido")

    # Mostrar todos los emails con sus pistas de dominio
    print()
    print("📊 Pistas de dominio para todos los emails de muestra:")
    print(f"{'Archivo':<50} {'Prestamista detectado':<35} {'Fuente'}")
    print("-" * 100)
    for eml_file in eml_files:
        e = parse_eml_file(eml_file)
        hint, src = identify_lender_from_domains(e.sender_domain, e.to_domains, e.cc_domains)
        hint_str = hint[:33] + ".." if hint and len(hint) > 35 else (hint or "❌ sin match")
        print(f"{eml_file.name:<50} {hint_str:<35} {src}")

---
## Paso 5 — El clasificador LLM (Ollama)

Este es el componente central. El LLM recibe un **prompt estructurado** con:
- La Knowledge Base completa
- La pista de dominio (del paso anterior)
- El contenido del email

Y devuelve un JSON con la clasificación.

### ¿Cómo funciona el prompt?

```
"Eres un experto clasificador de emails de cumplimiento de seguros..."
  + Base de conocimiento (todos los prestamistas y sus waivers)
  + "La pista de dominio dice: JLL"
  + Contenido del email
  = Respuesta JSON con { lender, waiver_type, confidence_score, reasoning }
```

> **Nota:** Si Ollama no está corriendo, el notebook usa automáticamente  
> una respuesta simulada basada en la pista de dominio.

In [ ]:
import json
import re

# ======================================================
# PROMPT TEMPLATE — lo mismo que usa el sistema real
# ======================================================
CLASSIFICATION_PROMPT = """Eres un experto clasificador de emails de cumplimiento de seguros para AcentoPartners.

=== CONTEXTO CRÍTICO ===
Estos emails son RESPUESTAS del agente de seguros AL prestamista, por eso:
- El campo TO identifica al PRESTAMISTA (no el FROM)
- El campo FROM es el agente de seguros respondiendo

=== PISTA DE DOMINIO ===
Prestamista detectado por dominio: {domain_lender_hint}
Fuente: {domain_hint_source}
IMPORTANTE: Usa esta pista como evidencia fuerte, pero verifica contra el contenido.

=== BASE DE CONOCIMIENTO ===
{knowledge_base}

=== PRESTAMISTAS VÁLIDOS ===
{lender_list}

=== TIPOS DE WAIVER VÁLIDOS ===
{waiver_type_list}

=== INSTRUCCIONES ===
1. PRESTAMISTA: Identifica cuál prestamista envió este email.
2. WAIVER TYPE: El problema de cumplimiento principal (usa nombres exactos de la lista).
3. ISSUES SECUNDARIOS: Otros problemas mencionados en el email.
4. TRIGGER: Qué específicamente disparó este reclamo.
5. CONFIANZA: 0.0-1.0 según claridad del match con la base de conocimiento.
   - 0.85-1.0: Prestamista y waiver claros
   - 0.60-0.84: Uno de los dos es ambiguo
   - Menor a 0.60: Ninguno está claro

Responde SOLO con JSON válido:
{{"lender": "<nombre>", "waiver_type": "<tipo>", "secondary_issues": [], "trigger_description": "<trigger>", "confidence_score": 0.0, "reasoning": "<razonamiento>"}}

=== EMAIL A CLASIFICAR ===
De: {sender}
Para: {to_recipients}
CC: {cc_recipients}
Asunto: {subject}
Adjuntos: {attachments}

Cuerpo:
{body}
"""


def build_prompt(email: EmailData, domain_hint: str | None, hint_source: str) -> str:
    """Construye el prompt completo para el LLM."""
    lender_names = list(dict.fromkeys(e["lender"] for e in LENDER_WAIVER_MATRIX))
    waiver_types = [e["waiver_type"] for e in LENDER_WAIVER_MATRIX]

    return CLASSIFICATION_PROMPT.format(
        knowledge_base=get_knowledge_base_text(),
        lender_list="\n".join(f"- {n}" for n in lender_names),
        waiver_type_list="\n".join(f"- {w}" for w in waiver_types),
        domain_lender_hint=domain_hint or "DESCONOCIDO",
        domain_hint_source=hint_source,
        sender=email.sender or "desconocido",
        to_recipients=", ".join(email.to_recipients[:5]) or "(vacío)",
        cc_recipients=", ".join(email.cc_recipients[:5]) or "(ninguno)",
        subject=email.subject or "(sin asunto)",
        attachments=", ".join(email.attachment_names[:8]) or "ninguno",
        body=email.body_text[:5000] or "(cuerpo vacío)",
    )


# Ver cómo se ve el prompt (solo las primeras líneas)
if eml_files:
    prompt = build_prompt(email, domain_hint, hint_source)
    print(f"📝 Longitud del prompt: {len(prompt)} caracteres")
    print()
    print("--- Primeras 800 caracteres del prompt ---")
    print(prompt[:800])
    print("...")

In [ ]:
from dataclasses import dataclass

@dataclass
class ClassificationResult:
    """Resultado de clasificar un email."""
    lender: str
    waiver_type: str
    trigger_description: str
    confidence_score: float
    confidence_level: str          # 'high', 'medium', 'low'
    secondary_issues: list = field(default_factory=list)
    reasoning: str = ""
    # Enriquecido desde KB después:
    required_evidence_ops: str = ""
    required_evidence_insurance: str = ""
    documents_expected: str = ""
    waiver_pack: str = ""
    actions_to_automate: str = ""
    used_mock: bool = False        # True si se usó respuesta simulada


def _score_to_level(score: float) -> str:
    if score > 0.85:
        return "high"
    elif score > 0.60:
        return "medium"
    return "low"


def _parse_llm_response(raw: str, domain_hint: str | None) -> ClassificationResult:
    """Parsea la respuesta JSON del LLM."""
    cleaned = re.sub(r"^```json\s*", "", raw.strip())
    cleaned = re.sub(r"\s*```$", "", cleaned)
    data = json.loads(cleaned)

    score = max(0.0, min(1.0, float(data.get("confidence_score", 0.5))))
    secondary = data.get("secondary_issues", [])
    if isinstance(secondary, str):
        secondary = [secondary] if secondary else []

    return ClassificationResult(
        lender=data.get("lender", "UNKNOWN"),
        waiver_type=data.get("waiver_type", "UNKNOWN"),
        trigger_description=data.get("trigger_description", ""),
        confidence_score=score,
        confidence_level=_score_to_level(score),
        secondary_issues=secondary,
        reasoning=data.get("reasoning", ""),
    )


def _mock_classify(email: EmailData, domain_hint: str | None) -> ClassificationResult:
    """
    Clasificación simulada — usada cuando Ollama no está disponible.
    Usa la pista de dominio y palabras clave del asunto.
    """
    lender = domain_hint or "UNKNOWN"
    subject_lower = (email.subject or "").lower()

    # Detectar waiver type por palabras clave en asunto
    if "assault" in subject_lower or "a&b" in subject_lower or "ab coverage" in subject_lower:
        waiver_type = "Assault & Battery (A&B) sublimit"
        score = 0.88
    elif "sexual abuse" in subject_lower or "sam" in subject_lower or "molestation" in subject_lower:
        waiver_type = "Sexual Abuse & Molestation (SAM)"
        score = 0.87
    elif "equipment breakdown" in subject_lower or "eb" in subject_lower:
        waiver_type = "Equipment Breakdown (EB) limit"
        score = 0.86
    elif "non-compliance" in subject_lower or "full policy" in subject_lower:
        waiver_type = "Full Policy Package timing"
        score = 0.85
    elif "freddie" in subject_lower or "wording" in subject_lower:
        waiver_type = "Additional Insured/Mortgagee wording"
        score = 0.84
    elif "acord" in subject_lower or "payment" in subject_lower:
        waiver_type = "ACORD-gate for payment & Umbrella clarity"
        score = 0.83
    elif "invoice" in subject_lower or "address" in subject_lower or "excess" in subject_lower:
        waiver_type = "Invoice components (Excess/Terrorism) & Address"
        score = 0.82
    elif "ol" in subject_lower or "bi" in subject_lower or "structure" in subject_lower:
        waiver_type = "OL / BI / EPI specifics"
        score = 0.81
    elif "security" in subject_lower or "assessment" in subject_lower:
        waiver_type = "Assault & Battery (A&B) sublimit"
        score = 0.75
    else:
        waiver_type = "Multi-issue compliance (A&B / SAM / BI / OL)"
        score = 0.55

    # Si no detectamos prestamista, bajar confianza
    if lender == "UNKNOWN":
        score = min(score, 0.45)

    return ClassificationResult(
        lender=lender,
        waiver_type=waiver_type,
        trigger_description=f"[MOCK] Clasificación basada en dominio y palabras clave del asunto",
        confidence_score=score,
        confidence_level=_score_to_level(score),
        reasoning=f"[Respuesta simulada] Dominio: {domain_hint}, asunto: {email.subject}",
        used_mock=True,
    )


print("✅ Funciones de clasificación definidas")

In [ ]:
import urllib.request
import json as _json

OLLAMA_AVAILABLE = False
OLLAMA_MODEL = "llama3.1:8b"
OLLAMA_HOST = "http://localhost:11434"

# Verificar con HTTP directo (evita el problema del event loop del SDK)
try:
    with urllib.request.urlopen(f"{OLLAMA_HOST}/api/tags", timeout=3) as resp:
        data = _json.loads(resp.read())
        available_models = [m["name"] for m in data.get("models", [])]

    if any(OLLAMA_MODEL in m for m in available_models):
        OLLAMA_AVAILABLE = True
        print(f"✅ Ollama disponible — modelo '{OLLAMA_MODEL}' listo")
        print(f"   Modelos instalados: {available_models}")
    else:
        print(f"⚠️  Ollama conectado pero modelo '{OLLAMA_MODEL}' no instalado")
        print(f"   Para instalarlo: ollama pull {OLLAMA_MODEL}")
        print(f"   Modelos disponibles: {available_models}")
        print("⚠️ Usando clasificación simulada (mock)")

except Exception as e:
    print(f"⚠️  Ollama no disponible: {e}")
    print("   👉 Usando clasificación simulada (mock) — resultados aproximados")


async def classify_email(email: EmailData) -> ClassificationResult:
    """
    Clasifica un email usando:
    1. Detección de dominio (pista fuerte)
    2. LLM Ollama (si disponible) o mock (si no)
    3. Enriquecimiento con Knowledge Base
    """
    from ollama import AsyncClient as OllamaClient

    # Paso A: detectar prestamista por dominio
    domain_hint, hint_source = identify_lender_from_domains(
        email.sender_domain, email.to_domains, email.cc_domains
    )

    # Paso B: clasificar con LLM o mock
    if OLLAMA_AVAILABLE:
        prompt = build_prompt(email, domain_hint, hint_source)
        client = OllamaClient(host=OLLAMA_HOST)
        response = await client.chat(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.1, "num_predict": 600},
            format="json",
        )
        result = _parse_llm_response(response.message.content, domain_hint)
    else:
        result = _mock_classify(email, domain_hint)

    # Paso C: enriquecer con datos de la Knowledge Base
    kb_entry = find_matching_entry(result.lender, result.waiver_type)
    if kb_entry:
        result.required_evidence_ops       = kb_entry["evidence_required_ops"]
        result.required_evidence_insurance  = kb_entry["evidence_required_insurance"]
        result.documents_expected           = kb_entry["documents_expected"]
        result.waiver_pack                  = kb_entry["waiver_pack"]
        result.actions_to_automate          = kb_entry["actions_to_automate"]

    return result


print("✅ Función classify_email lista")


---
## Paso 6 — Clasificar un email individual

Ahora aplicamos el clasificador a un email real y vemos el resultado completo.

In [ ]:
# Clasificar el primer email
if eml_files:
    print(f"🔄 Clasificando: {email.filename}")
    print(f"   Asunto: {email.subject}")
    print()

    result = await classify_email(email)

    # Indicador visual de confianza
    nivel_icons = {"high": "🟢", "medium": "🟡", "low": "🔴"}
    mock_note = " [SIMULADO]" if result.used_mock else " [LLM real]"

    print("=" * 60)
    print(f"📊 RESULTADO DE CLASIFICACIÓN{mock_note}")
    print("=" * 60)
    print(f"🏦 Prestamista:     {result.lender}")
    print(f"📋 Tipo de Waiver:  {result.waiver_type}")
    print(f"{nivel_icons.get(result.confidence_level, '⚪')} Confianza:       {result.confidence_score:.2f} ({result.confidence_level})")
    print(f"⚡ Trigger:         {result.trigger_description}")
    print()
    if result.secondary_issues:
        print(f"⚠️  Issues secundarios: {', '.join(result.secondary_issues)}")
        print()
    print("📚 Enriquecimiento desde Knowledge Base:")
    print(f"   Evidencia Ops:       {result.required_evidence_ops[:100]}..." if len(result.required_evidence_ops) > 100 else f"   Evidencia Ops:       {result.required_evidence_ops}")
    print(f"   Evidencia Seguros:   {result.required_evidence_insurance[:100]}..." if len(result.required_evidence_insurance) > 100 else f"   Evidencia Seguros:   {result.required_evidence_insurance}")
    print(f"   Documentos a enviar: {result.documents_expected}")
    print()
    print(f"📦 WaiverPack a armar:")
    for doc in result.waiver_pack.split(","):
        print(f"   • {doc.strip()}")
    print()
    print(f"🤖 Razonamiento: {result.reasoning[:300]}")

---
## Paso 7 — Base de datos simulada

En el sistema real se usa SQLite con SQLAlchemy async.  
Aquí lo simulamos con una lista de diccionarios en memoria para entender la estructura.

Cada registro guarda:
- Metadatos del email (remitente, asunto, fecha)
- Resultado de la clasificación (prestamista, waiver, confianza)
- Estado del workflow (clasificado → revisado → corregido)

In [ ]:
import uuid
from datetime import datetime, timezone

# Base de datos simulada (en producción: SQLite/PostgreSQL)
DB: list[dict] = []


def save_classification(email: EmailData, result: ClassificationResult) -> dict:
    """
    Guarda un resultado de clasificación en la 'base de datos'.
    En producción esto es un INSERT en SQLite via SQLAlchemy.
    """
    record = {
        # Identificador único
        "id": str(uuid.uuid4())[:8],
        # Metadatos del email
        "source": email.source,
        "filename": email.filename,
        "message_id": email.message_id,
        "subject": email.subject,
        "sender": email.sender,
        "received_date": str(email.received_date),
        "body_preview": email.body_text[:500],
        # Resultado de clasificación
        "lender": result.lender,
        "waiver_type": result.waiver_type,
        "trigger_description": result.trigger_description,
        "confidence_score": result.confidence_score,
        "confidence_level": result.confidence_level,
        "secondary_issues": result.secondary_issues,
        # Enriquecimiento KB
        "documents_expected": result.documents_expected,
        "waiver_pack": result.waiver_pack,
        "required_evidence_ops": result.required_evidence_ops,
        "required_evidence_insurance": result.required_evidence_insurance,
        # Estado del workflow
        "status": "classified",       # classified → reviewed → corrected
        "reviewed_by": None,
        "corrected_lender": None,
        "corrected_waiver_type": None,
        "correction_notes": None,
        # Timestamps
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    DB.append(record)
    return record


# Guardar el resultado del email anterior
if eml_files:
    record = save_classification(email, result)
    print(f"✅ Guardado en DB — ID: {record['id']}")
    print(f"   Status: {record['status']}")
    print(f"   Registros en DB: {len(DB)}")

---
## Paso 8 — Pipeline completo: clasificar TODOS los emails

Ahora ejecutamos el pipeline end-to-end sobre todos los archivos `.eml` de muestra.

In [ ]:
async def run_full_pipeline(eml_files: list[Path]) -> list[dict]:
    """Corre el pipeline completo sobre todos los archivos .eml."""
    results = []
    print(f"🚀 Procesando {len(eml_files)} emails...\n")

    for i, eml_file in enumerate(eml_files, 1):
        print(f"[{i}/{len(eml_files)}] {eml_file.name[:55]}")
        try:
            # 1. Parsear
            email_data = parse_eml_file(eml_file)
            # 2. Clasificar
            classification = await classify_email(email_data)
            # 3. Guardar
            record = save_classification(email_data, classification)

            nivel_icons = {"high": "🟢", "medium": "🟡", "low": "🔴"}
            icon = nivel_icons.get(classification.confidence_level, "⚪")
            mock_note = "[mock]" if classification.used_mock else "[llm]"
            print(f"       {icon} {mock_note} {classification.lender[:30]:<30} | {classification.waiver_type[:35]:<35} | conf: {classification.confidence_score:.2f}")
            results.append(record)
        except Exception as e:
            print(f"       ❌ Error: {e}")

    print(f"\n✅ Pipeline completado: {len(results)}/{len(eml_files)} exitosos")
    return results


# Limpiar DB y correr pipeline completo
DB.clear()
all_results = await run_full_pipeline(eml_files)

---
## Paso 9 — Human-in-the-Loop: Cola de revisión y correcciones

No todas las clasificaciones son confiables. El sistema tiene un flujo de revisión humana:

- **Confianza alta (>0.85)** → se procesa automáticamente
- **Confianza media (0.60–0.85)** → va a la cola de revisión
- **Confianza baja (<0.60)** → va a la cola de revisión con prioridad

El operador puede **aprobar** o **corregir** cada clasificación.

In [ ]:
def get_review_queue() -> list[dict]:
    """
    Retorna los registros pendientes de revisión humana:
    - status == 'classified' (no revisado aún)
    - confidence_level == 'medium' o 'low'
    Ordenados de menor a mayor confianza (los menos seguros primero).
    """
    queue = [
        r for r in DB
        if r["status"] == "classified"
        and r["confidence_level"] in ("medium", "low")
    ]
    return sorted(queue, key=lambda x: x["confidence_score"])


def approve_classification(record_id: str, reviewed_by: str = "operator") -> dict:
    """Aprueba una clasificación como correcta."""
    for record in DB:
        if record["id"] == record_id:
            record["status"] = "reviewed"
            record["reviewed_by"] = reviewed_by
            return record
    raise ValueError(f"Registro {record_id} no encontrado")


def correct_classification(
    record_id: str,
    corrected_lender: str,
    corrected_waiver_type: str,
    reviewed_by: str = "operator",
    notes: str = "",
) -> dict:
    """
    El operador corrige una clasificación incorrecta.
    Guarda los valores originales y re-enriquece con la KB.
    """
    for record in DB:
        if record["id"] == record_id:
            # Guardar valores originales
            record["corrected_lender"] = corrected_lender
            record["corrected_waiver_type"] = corrected_waiver_type
            record["reviewed_by"] = reviewed_by
            record["correction_notes"] = notes
            record["status"] = "corrected"

            # Actualizar clasificación con los valores corregidos
            record["lender"] = corrected_lender
            record["waiver_type"] = corrected_waiver_type

            # Re-enriquecer desde KB con los valores corregidos
            kb_entry = find_matching_entry(corrected_lender, corrected_waiver_type)
            if kb_entry:
                record["documents_expected"] = kb_entry["documents_expected"]
                record["waiver_pack"] = kb_entry["waiver_pack"]
                record["required_evidence_ops"] = kb_entry["evidence_required_ops"]
                record["required_evidence_insurance"] = kb_entry["evidence_required_insurance"]
            return record
    raise ValueError(f"Registro {record_id} no encontrado")


# Ver la cola de revisión
queue = get_review_queue()
print(f"📋 Cola de revisión humana: {len(queue)} registros pendientes")
print(f"   (de {len(DB)} clasificaciones totales)")
print()

if queue:
    print(f"{'ID':<10} {'Confianza':<10} {'Nivel':<8} {'Prestamista':<32} {'Waiver Type'}")
    print("-" * 100)
    for r in queue:
        nivel_icons = {"high": "🟢", "medium": "🟡", "low": "🔴"}
        icon = nivel_icons.get(r['confidence_level'], '⚪')
        print(f"{r['id']:<10} {r['confidence_score']:<10.2f} {icon} {r['confidence_level']:<6} {r['lender'][:30]:<32} {r['waiver_type'][:40]}")

In [ ]:
# Simular acciones del operador sobre la cola de revisión
queue = get_review_queue()

if len(queue) >= 2:
    # Aprobar el primero
    approved = approve_classification(queue[0]["id"], reviewed_by="maria.garcia")
    print(f"✅ Aprobado: ID={approved['id']} → status={approved['status']}")

    # Corregir el segundo
    corrected = correct_classification(
        record_id=queue[1]["id"],
        corrected_lender="JLL (Insurance Servicing)",
        corrected_waiver_type="Assault & Battery (A&B) sublimit",
        reviewed_by="carlos.lopez",
        notes="El prestamista real es JLL, el dominio en TO lo confirma",
    )
    print(f"✏️  Corregido: ID={corrected['id']}")
    print(f"   Prestamista actualizado: {corrected['lender']}")
    print(f"   Waiver actualizado:      {corrected['waiver_type']}")
    print(f"   Revisado por:            {corrected['reviewed_by']}")
    print(f"   Notas:                   {corrected['correction_notes']}")
elif len(queue) == 1:
    approved = approve_classification(queue[0]["id"], reviewed_by="operador")
    print(f"✅ Aprobado: ID={approved['id']}")
else:
    print("ℹ️  No hay registros en cola de revisión (todos tienen confianza alta)")

---
## Paso 10 — Estadísticas del sistema

El sistema calcula métricas agregadas de todas las clasificaciones.  
En el sistema real esto se obtiene con queries SQL GROUP BY.  
Aquí lo calculamos directamente desde la lista en memoria.

In [ ]:
from collections import Counter

def get_stats() -> dict:
    """Calcula estadísticas de todas las clasificaciones en DB."""
    total = len(DB)
    if total == 0:
        return {"total": 0}

    by_lender = Counter(r["lender"] for r in DB)
    by_waiver = Counter(r["waiver_type"] for r in DB)
    by_confidence = Counter(r["confidence_level"] for r in DB)
    by_status = Counter(r["status"] for r in DB)

    avg_confidence = sum(r["confidence_score"] for r in DB) / total

    reviewed_total = by_status.get("reviewed", 0) + by_status.get("corrected", 0)
    correction_rate = by_status.get("corrected", 0) / reviewed_total if reviewed_total > 0 else 0.0

    return {
        "total_clasificados": total,
        "por_prestamista": dict(by_lender.most_common()),
        "por_tipo_waiver": dict(by_waiver.most_common()),
        "por_nivel_confianza": dict(by_confidence),
        "por_status": dict(by_status),
        "confianza_promedio": round(avg_confidence, 3),
        "tasa_corrección": round(correction_rate, 3),
    }


stats = get_stats()

print("=" * 60)
print("📊 ESTADÍSTICAS DEL SISTEMA")
print("=" * 60)
print(f"Total clasificados:    {stats['total_clasificados']}")
print(f"Confianza promedio:    {stats['confianza_promedio']:.1%}")
print(f"Tasa de corrección:    {stats['tasa_corrección']:.1%}")
print()

print("Por nivel de confianza:")
for level, count in sorted(stats['por_nivel_confianza'].items()):
    icons = {"high": "🟢", "medium": "🟡", "low": "🔴"}
    bar = "█" * count
    print(f"  {icons.get(level,'⚪')} {level:<8}: {bar} ({count})")

print()
print("Por status:")
status_icons = {"classified": "📋", "reviewed": "✅", "corrected": "✏️"}
for status, count in stats['por_status'].items():
    print(f"  {status_icons.get(status,'⚪')} {status:<12}: {count}")

print()
print("Por prestamista:")
for lender, count in stats['por_prestamista'].items():
    bar = "█" * count
    print(f"  {lender[:45]:<45}: {bar} ({count})")

print()
print("Por tipo de waiver:")
for waiver, count in stats['por_tipo_waiver'].items():
    bar = "█" * count
    print(f"  {waiver[:50]:<50}: {bar} ({count})")

---
## Paso 11 — Ver todos los resultados en tabla

Resumen visual de todas las clasificaciones procesadas.

In [ ]:
nivel_icons = {"high": "🟢", "medium": "🟡", "low": "🔴"}
status_icons = {"classified": "📋", "reviewed": "✅", "corrected": "✏️"}

print(f"{'#':<3} {'ID':<10} {'Asunto':<45} {'Prestamista':<30} {'Conf':>5} {'Niv':<6} {'Status'}")
print("-" * 120)

for i, r in enumerate(DB, 1):
    icon = nivel_icons.get(r['confidence_level'], '⚪')
    st_icon = status_icons.get(r['status'], '⚪')
    subject = (r['subject'] or '')[:43]
    lender = r['lender'][:28]
    print(
        f"{i:<3} {r['id']:<10} {subject:<45} {lender:<30} "
        f"{r['confidence_score']:>5.2f} {icon} {r['confidence_level']:<4} {st_icon} {r['status']}"
    )

---
## Resumen — Arquitectura completa

```
┌─────────────────────────────────────────────────────────────────┐
│                    PIPELINE DE CLASIFICACIÓN                     │
├─────────────────────────────────────────────────────────────────┤
│                                                                   │
│  📧 Email (.eml / Outlook)                                       │
│       │                                                           │
│       ▼                                                           │
│  📋 PARSER  →  extrae: FROM, TO, CC, body, adjuntos             │
│       │                                                           │
│       ▼                                                           │
│  🔍 DETECCIÓN DE DOMINIO  →  TO > CC > FROM  →  pista lender    │
│       │                                                           │
│       ▼                                                           │
│  🤖 LLM (Ollama llama3.1:8b)                                     │
│       │  prompt = KB + pista de dominio + email                  │
│       │  output = { lender, waiver_type, confidence, reasoning } │
│       │                                                           │
│       ▼                                                           │
│  📚 ENRIQUECIMIENTO KB  →  agrega: docs, waiver_pack, evidencia  │
│       │                                                           │
│       ▼                                                           │
│  💾 BASE DE DATOS  →  status: classified                         │
│       │                                                           │
│       ▼                                                           │
│  👤 HUMAN-IN-THE-LOOP                                            │
│       ├── alta confianza  →  auto-procesado                      │
│       └── media/baja      →  cola de revisión                    │
│               ├── aprobar  →  status: reviewed                   │
│               └── corregir →  status: corrected + re-enriquece   │
│                                                                   │
│  (futuro) 📤 RESPUESTA AUTOMÁTICA vía Graph API + adjuntos       │
└─────────────────────────────────────────────────────────────────┘
```

### Archivos del proyecto real

| Componente | Archivo real | Célda en este notebook |
|---|---|---|
| Knowledge Base | `app/core/knowledge_base.py` | Paso 1, 2 |
| Email Parser | `app/services/email_parser/parser.py` | Paso 3 |
| Domain Detection | `app/core/knowledge_base.py` | Paso 4 |
| LLM Classifier | `app/services/classifier/llm_classifier.py` | Paso 5 |
| Orchestrator | `app/services/orchestrator.py` | Paso 8 |
| Database | `app/models/database.py` | Paso 7 |
| Review Queue | `app/api/routes.py` | Paso 9 |
| Stats | `app/services/orchestrator.py` | Paso 10 |
| API REST (FastAPI) | `app/main.py`, `app/api/routes.py` | *(simulado aquí con funciones directas)* |